In [1]:

import ipynbname
from jupyter_lab_notebook_toc_utils import generate_toc, display_toc
toc = generate_toc(ipynbname.name() + ".ipynb", add_numbering=True)
display_toc(toc)

**Table of Contents**<br/>
&nbsp;&nbsp;&nbsp;&nbsp; [**1.** **Overview**](#Overview)<br/>
&nbsp;&nbsp;&nbsp;&nbsp; [**2.** **Major Topics**](#Major-Topics)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**2.1.** Encoders and Decoders](#Encoders-and-Decoders)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**2.2.** Sequence to Sequence (seq2seq)](#Sequence-to-Sequence-%28seq2seq%29)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**2.3.** The Reordering Problem](#The-Reordering-Problem)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**2.4.** Vanishing/Exploding Gratient Problem](#Vanishing/Exploding-Gratient-Problem)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**2.4.1.** What is a gradient and how does it explode?](#What-is-a-gradient-and-how-does-it-explode?)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**2.4.2.** Relation with Short Term and Long Term Memory](#Relation-with-Short-Term-and-Long-Term-Memory)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**2.4.3.** Potential Solutions To The Gradient Problem](#Potential-Solutions-To-The-Gradient-Problem)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**2.5.** The Fixed Vector Problem](#The-Fixed-Vector-Problem)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**2.6.** Solving the Fixed Vector Problem: Variable Vectors, Context Vectors, Prallel Training, and Attention](#Solving-the-Fixed-Vector-Problem:-Variable-Vectors,-Context-Vectors,-Prallel-Training,-and-Attention)<br/>
&nbsp;&nbsp;&nbsp;&nbsp; [**3.** **History**](#History)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**3.1.** Ancient Times - 1630s](#Ancient-Times---1630s)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**3.2.** Machine Translation (1940s - 2000s)](#Machine-Translation-%281940s---2000s%29)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**3.2.1.** Rule Based Machine Translation (1940s - 1980s)](#Rule-Based-Machine-Translation-%281940s---1980s%29)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**3.2.2.** Statistical Machine Translation (1980s - 2000s)](#Statistical-Machine-Translation-%281980s---2000s%29)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**3.2.3.** Neural Machine Translation (1990s - Present)](#Neural-Machine-Translation-%281990s---Present%29)

# Overview

In this notebook we aim to discuss the topic of encoder-decoders within Machine Learning. We will understand how they are built, what they can do, and the history behind their discovery and evolution.

# Major Topics

## Encoders and Decoders

Generally speaking an encoder-decoder is a two part system for translating text from one form, into an intermediate form, and then translating that intermediate form into a final form. This paradigm can be used for a wide variety of applications including encryption, translation, summarization, and question and answer.

Within the context of machine learning, encoder-decoder architectures are, again two part systems, consisting of two neural networks which discretely provide the encoder and decoder functionality.

## Sequence to Sequence (seq2seq)

A quick note on teminology: 

At a certain point within the machine translation field, people started thinking about translation as a mapping between a sequence of text in different languages. For example Tomáš Mikolov's 2012 [PhD thesis](https://www.fit.vut.cz/study/phd-thesis-file/283/283.pdf).

The first [paper](https://arxiv.org/abs/1511.06391) I could find which uses the term seq2seq was published in Nov 2015 by Vinyals, Benjio, and Kudler. It describes seq2seq as an existing framework which employs the chain rule (back propogation and/or maximum likelihood) and implicitly characterizes the framework as having an encoder-decoder architecture.

I bring this up because it starts to appear in papers, surverys, and summaries and I figure we should introduce it asap.

## The Reordering Problem

According to this [paper](https://qmro.qmul.ac.uk/xmlui/handle/123456789/2517):
> The reordering problem in SMT originates from the fact that not all the words in a sentence can be consecutively translated. This means words must be skipped and be translated out of their order in the source sentence to produce a fluent and grammatically correct sentence in the target language. The main reason that reordering is needed is the fundamental word order differences between languages.

For example, in english we might say "the white house" while in french we might say "la maison blanche" (directy translated as "the house white".

> Therefore, reordering becomes a more dominant issue, the more source and target languages are structurally different.


According to this [article](https://direct.mit.edu/coli/article/42/2/163/1530/A-Survey-of-Word-Reordering-in-Statistical-Machine) published in 2016:

> Word reordering is one of the most difficult aspects of statistical machine translation (SMT), and an important factor of its quality and efficiency. Despite the vast amount of research published to date, the interest of the community in this problem has not decreased, and no single method appears to be strongly dominant across language pairs. Instead, the choice of the optimal approach for a new translation task still seems to be mostly driven by empirical trials.
> 
> ...
> 
> the core SMT methods (Brown et al. 1990, 1993; Berger et al. 1996; Koehn, Och, and Marcu 2003) learn direct correspondences between source and target language from collections of translated sentences, without the need for abstract linguistic representations. The main advantages of SMT are versatility and cost-effectiveness: In principle, the same modeling framework can be applied to any pair of languages with minimal engineering effort, given sufficient amounts of translation data. However, experience in a diverse range of language pairs has revealed that this form of modeling is highly sensitive to structural differences between source and target language, particularly at the level of word order.
> 
> ...
> 
> Searching for the overall best translation throughout the space of all possible reorderings is, however, computationally intractable (Knight 1999). This crucial fact has motivated an impressive amount of research around two inter-related questions: namely, how to effectively restrict the set of allowed word permutations and how to detect the best permutation among them.
> ...
>
>  String-based SMT (Sections 2.1 and 2.2) treats translation as a sequential task: The target sentence is built from left to right while the input units are visited in different orders and no dependencies other than word adjacency are considered. ... Tree-based SMT (Section 2.3) posits the existence of a tree structure to explain translation as a hierarchical process and to capture dependencies among non-adjacent text units. Problem decomposition is therefore based on this structure: An optimal translation is sought for each word span corresponding to a node in the tree, from the leaves up to the root. Whereas string-based SMT has to search over all input permutations that do not violate some general reordering constraints, tree-based SMT considers only those permutations that result from transforming a given tree representing the input sentence (as for example permuting each node's children).
> ...
> 



## Vanishing/Exploding Gratient Problem

One of the most common problems when working with Deep Neural Networks is referred to as the Vanishing and/or Exploding Gradient Problem. 

This problem prevented NNs from being used in encoder-decoder architectures or machine translation for some time.

### What is a gradient and how does it explode?
So what is a gradient, and what does it have to do with Deep Neural Networks? 

Recall that when training a neural network, we perform forward propogation followed by back propogation. With forward propogation, the input data flows through the input layers, then the hidden layers, then the output layer of the the neural network in effect generating the training output. Then based on the correct output, backpropogation occurs; moving in the reverse direction as forwad propogation, the back propogation process adjusts the weights in the network yielding a more accurate prediction given the same input. In order for back propogation to work, there must be some process employed which will optimize the weights to yield the best result. And this is where the relation with the gradient comes in. The mathematic process underlying the backpropogation is typically gradient discent; an optimiazation algorithm which uses the gradient to find an assumed local minimal.

OK, so we now know what a gradient is, what's the vanishing or exploding gradient problem?

The vanishing gradient problem arrises when the gradient used by gradient dissent approaches zero before back propogation has reached the upper levels. Normally, the gradient approaches zero as we approach the local minima, but in scenarios where we approach zero in the deeper layers, the network is not actually getting claibrated. The wieghts in the higher level will remain reletively unchanged resulting in poor predictive performance. The exploding gradient is the opposite: in this scenario the greadient keeps growing which in turn again prevents convergence and model calibration.

Some texts will refer to the neural network or its training process as being, or becoming, unstable when the underlying model suffers from the vanishing/exploding gradient problem.

In 1991 Hochreiter published his [doctoral thesis](http://www.bioinf.jku.at/publications/older/3804.pdf) which provided an analysis of the gradient problem. [According to Schmidhuber](https://people.idsia.ch/~juergen/fundamentaldeeplearningproblem.html), the professor and thesis supervisor, Hochreiter's paper formally showed that deep neural networks are hard to train, because they suffer from the now famous problem of vanishing or exploding gradients.

### Relation with Short Term and Long Term Memory

Thinking abstractly about how the NNs work; we can think of each layer as providing a memory of the original input. As we move deeper into the neural network's layers, we can think of the memory as transitioning from short term to long term. Additionally, as each layer assigns a weight to the prior layer's activations, we can think of this weight as part of the memory mechanism as well. Moving deeper into the model's layers, the memory becomes less clear and starts to be impacted by the bias (weights) of the model.

In some cases, as mentioned in this [StackOverflow article](https://stackoverflow.com/questions/54286472/q-why-long-short-term-memorylstm-is-called-as-a-long-and-short-both-type-of-m), a NN layer may attach a low weight to an input whech gradually results in a weight of zero. In this way the NN is "forgetting" the input. The problem, is that the forgotten information may be important or relevant relevant to the prediction rendering it unable to "remember".

> Think of for example a piece of text. "Barnie is a big red dog, with little ears and a long black tail. He is 12 years old". If your task was to figure out what "He" refers to in the second sentence, you would send this data into an LSTM network, and it would analyze each word individually. The calculations for a single word is the Short-Term Memory. However the calculations of each word (the hidden state), ... is passed on and included when analyzing the next word. ... therefore storing the Short-Term data (calculations of individual word) over Long periods of time (passing the hidden states to the next word).

The trick here is that there needs to be a balance between short term and long term memory as not all information is relevant. Additionally, the longer term your memory the more complex the model and the more noise potentially introduced into the system.

In response to this problem, new model architectures are proposed which provided enhanced memory capabilities. We will talk about these in more detail later in this article. Additionally, new attention mechanisms are proposed which will filter out irrelevant information provided by the memory mechanisms. For more information see the [Attention notebook](Attention.ipyb).

So we see that the vanishing or exploding gradient problems are related to the Short Term vs Long Term Memory problem.

### Potential Solutions To The Gradient Problem

In order to prevent this from happening, one solution is initializing weights to random values. This is a common approach used when searching via gradient dissent. By initializing the weights to a random value, the search algorithm has a better change of converging. A good read on the subject can be found [here](https://www.comet.com/site/blog/vanishing-exploding-gradients-in-deep-neural-networks/#:~:text=Exploding%20is%20the%20opposite%20of,Network%2C%20not%20the%20activation%20function).


In addition to this approach, Schmidhuber [states](https://people.idsia.ch/~juergen/fundamentaldeeplearningproblem.html) (in 2013) that currently there are four methods currently known to overcome the vanishing gradient problem:

1. Unsupervised pre-training for a hierarchy of (recurrent) neural networks

    Accoding to Schmidhuber, "This greatly facilitated subsequent supervised credit assignment through back-propagation."
   
   - ([1](https://people.idsia.ch/~juergen/fundamentaldeeplearningproblem.html)) Sepp Hochreiter's Fundamental Deep Learning Problem
   - (2) J. Schmidhuber. Learning complex, extended sequences using the principle of history compression, Neural Computation, 4(2):234-242, 1992 (based on TR FKI-148-91, 1991).

2. LSTM-like networks

    These types of networks avoid the problem through special architecture unaffected by it

3. Faster GPU-based

    Thse do not solve the problem, but reduce the impact of the problem so that despite the poorly trained network, the models still perform within the bounds of practicalitly.

4. Using alternat optimization algorithms besides gradient Dissent

   The space of NN weights can be searched by algorithms which do not rely on gradient matrices or gradient dissent.

Additionally we will see that newer architectures are able to use gradients reliably and produce better results. For example ([Mikolav et al. 2014](https://arxiv.org/abs/1412.7753)).

## The Fixed Vector Problem

While the LSTM model was the first to overcome the vanishing/exploding gradient problem. However, its introduces a new problem into the encoder-decoder field: the fixed vector problem. Because the LSTM encoder uses a vector with fixed dimensionality to encode information passed to the decoder, there is a limitation on the density of information and the amount of memory that the decoder has access to.

This limitation of using a fixed vector continues for some time. For example in September 2014, Sutskever et al. published a [paper](https://arxiv.org/abs/1409.3215) which presents a new model which is still bound by the same limitation.

The problem and it's solution is discussed in the complementary notebook on [Word Embeddings](#Word%20Embeddings.ipynb#Traditional-Word-Embeddings).

## Solving the Fixed Vector Problem: Variable Vectors, Context Vectors, Prallel Training, and Attention

The solution however, began to be uncovered when the following monumental advancements were made:

First, In June 2014, Cho, Benjio, et al. publish a [paper](https://arxiv.org/abs/1406.1078) propose a novel neural network model called RNN Encoder-Decoder in which the encoder and decoder are both RNNs and are trained in parallel. The paper considers the LSTM unit as one of many activation functions that can "plug into" an RNN and suggests that it's hidden unit is much simpler to compute. It notes that the hidden unit performs a similar function to the LSTM unit but "may also be considered an adaptive variant of a leaky-integration
unit (Bengio et al., 2013)".

Then, in the same month, June 2014, Mnih et al. publish a [paper](https://arxiv.org/abs/1406.6247) introducing the concept of Reccurrant Attention Model (RAM) which I believe is the first use of attention within RNNs. In this architecture, the model automatically adjusts the "bandwidth" or the "vector length" being observed durign the training process.

And in Sept 2014, Bahdanau, Cho, Benjio. published a [paper](https://arxiv.org/abs/1409.0473) propose an encoder-decoder which is not using a fixed length intermediary but a:

> an extension to the encoder–decoder model which learns to align and translate jointly. Each time the proposed model generates a word in a translation, it (soft-)searches for a set of positions in a source sentence where the most relevant information is concentrated. The model then predicts a target word based on the context vectors associated with
these source positions and all the previous generated target words.
>
> The most important distinguishing feature of this approach from the basic encoder–decoder is that
it does not attempt to encode a whole input sentence into a single fixed-length vector. Instead, it encodes the input sentence into a sequence of vectors and chooses a subset of these vectors adaptively while decoding the translation. This frees a neural translation model from having to squash all the information of a source sentence, regardless of its length, into a fixed-length vector. We show this allows a model to cope better with long sentences.

This paper introduces the concept of a Context Vector which replaces the prior fixed length intermediary representations.

Ultimately the individual solutions of variable Vectors, Context Vectors, Prallel Training, and Attention merge in the publication of the transformer architecture.

# History

The history detailed below focuses on advancements of the Encoder-decoder paradigm. Related topics like the usage of [Attention](Attention.ipynb) or [Word Embeddings](Word%20Embeddings.ipynb) are covered in their respective notebooks.

## Ancient Times - 1630s
The concept of encoding and decoding are nothing new. They have been used for thousands if not tens of thousands of years with respect to sending and receiving messages or recording information in documents. The basic idea of encoding is that the "natural form" of a text is not condusive to the task at hand. Maybe because we need to transmit the information through some physical mechanism which requires the information to be presented in a different format; for example, sending morris code over a wire. Or maybe because we want to keep the information secret; for example we may encrypt the data before saving it. Consequently, the process of decoding is simply the process of translating the encoded text back into it's original format.

The origins of the word come from the latin word *codex* which refers to a book of laws. Within the codex are the codes or laws for a particular subject. Thinking abstractly, in terms of a laungage, the codex defines how information is structured etc. within a language. Using the codex one can construct segments of text which are in compliance with the codex and thus are in-code or en-coded (from the old french).

Despite being used for thousands of years, from what I gather, the word encode first appeared in the 1930s

## Machine Translation (1940s - 2000s)
In the context of machine learning, the term's meaning is still consistent with it's historical usage.

According to [Zhang (2017) - History and Frontier of the Neural Machine Translation](https://syncedreview.com/2017/08/17/history-and-frontier-of-the-neural-machine-translation):

> Machine translation (MT) is utilizing the power of machines to do “automatic translation of text from one natural language (the source language) to another (the target language)” [1]. The idea of doing translation using machines was first raised by Warren Weaver in 1949. For a long time (1950s~1980s), machine translation was done through the study of the linguistic information about the source and target languages, generating translations based on the dictionaries and grammars, which is called rule-based machine translation (RBMT). With the development of Statistics, statistical models started to be applied to machine translation, which generates translations based on the analysis of bilingual text corpus. This method is known as the statistical machine translation (SMT), which gained better performance than RBMT and dominated the field from the 1980s to 2000s. In the year of 1997, Ramon Neco and Mikel Forcada came up with the idea of using “encoder-decoder” structure to do machine translations [2].
> 
> References:
> 
> - (1) Russell, S. & Norvig, P. (1995). Artificial intelligence: a modern approach
> - (2) Neco, R. P., & Forcada, M. L. (1997, June). Asynchronous translations with recurrent neural nets. In Neural Networks, 1997., International Conference on (Vol. 4, pp. 2535-2540). IEEE.


Machine translation, specifically SMT had several issues to overcome however:

### Rule Based Machine Translation (1940s - 1980s)

### Statistical Machine Translation (1980s - 2000s)

### Neural Machine Translation (1990s - Present)

#### Hochreiter (1995, 1997) - Long Short-Term Memory (LSTM)

Long Short-Term Memory (LSTM) is a method of back propogation for Recurrent Neural Network (RNN) published in 1995 by Hochreiter under the supervision of Schmidhuber. This was followed up by an improved revision of the [paper](https://www.researchgate.net/publication/13853244_Long_Short-term_Memory) in 1997 which is typically sourced as the origin of the term.  It's notariaty primarily stems from the claim that it is able to overcome the vanishing gradient problem But in, addition, the paper boasts more stable and faster training times resulting in RNNs which outperform the previous generation.

Another term that pops up tangentially is the concept of distance. We might see long term memory and long distance memory used interchangibly. The basic relationship between the two terms comes from the way that LSTM and memory in general works in the context of machine translation. An input sequence of text is provided and the mechanism generating the translation needs to "keep in mind" a certain trailing set of information. The idea with memory is that it's effectively storing an array of sequence tokens, the more recently observed ones having occured in the short term and being indexed closer to, with a shorter distance from, the current token being evalutated.

Another problem that LSTM claims to solve is that it extends the distance or term of the memory while allowing the model to filter out the noise (unimportant tokens in the input sequence).

Since then the original LSTM model has lead to a whole sub-family of models. More information on this topic can be found [here](https://mindmajix.com/what-is-lstm).

Additionally, because of it's complexity, researchers have attempted to obtain similar results with reduced complexity. Reading between the lines in some of these white papers, it looks like the math/architecture in LSTM is quite complicated compared to the modern front runners in the space. 


While the LSTM model was the first to overcome the vanishing/exploding gradient problem it introduced a new problem: the fixed vector problem. The approach uses a fixed dimensionality of the intermediary sequence being passed between the encoder and decoder.

This problem limitation of using a fixed vector continues for some time. For example in September 2014, Sutskever et al. published a paper which presents a new model which is still bound by the same limitation.

#### Benjio (2003) - Birth of NMT

In 2003, a group of researchers at the University of Montreal led by Yoshua Bengio [published](https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf) *A Neural Probabilistic Language Model* which proposed a novel language model which uses neural networks to construct the a representation of word meaning. The authors claim that this approach out performs preious n-gram models popularized by SMT.

**Note**: This is not a method for encoder-decoder translation. Instead it is a paper about a mathematical representation of word meaning. This will later become known as an embedding vector (i.e. a word embedding, see the [Workd Embedings notebook](Word%20Embeddings.ipynb) for more details) which, as we will see, can be used by encoder-decoder based architectures.

Reading through the abstract, we see that the authors claim that the proposed approach will address the so called *curse of dimensionality*. Additionally the model has two important characteristics:
1. It relies of a *distributed representation for words* 
2. It defines a probability function for word sequences expressed as embeddings 

It is important to note that the vector representation aleviates some of the data sparsity issues of previous SMT approaches. This advancement cannot be understated. While non-sparse vectors were used previously in the field of Topic Modeling (discussed in the [Topic Modeling notebook](Topic%20Modeling.ipynb)) they did not rely on neural networks and they were not strictly being applied to machine translation.

#### Kalchbrenner et. al. (2013) - NMT with CNN to RNN Architecture

In 2013, Nal Kalchbrenner and Phil Blunsom [published](https://aclanthology.org/D13-1176.pdf) *Recurrent Continuous Translation Models* which proposes a new encoder-decoder architecture for machine translation which relies on the usage of a CNN encoder and a RNN decoder.

According to [Zhang (2017) - History and Frontier of the Neural Machine Translation](https://syncedreview.com/2017/08/17/history-and-frontier-of-the-neural-machine-translation):
> This model will encode a given source text into a continuous vector using Convolutional Neural Network (CNN), and then use Recurrent Neural Network (RNN) as the decoder to transform the state vector into the target language. Their work can be treated as the birth of the Neural Machine Translation (NMT), which is a method that uses deep learning neural networks to map among natural language.

[Zhang (2017)](https://syncedreview.com/2017/08/17/history-and-frontier-of-the-neural-machine-translation) goes on to note the proposed benefits of this NMT approach as compared to previous SMT approaches.

> NMT’s nonlinear mapping differs from the linear SMT models, and describes the semantic equivalence using the state vectors which connect encoder and decoder. In addition, the RNN is supposed to be capable of capturing information behind an infinite length of sentences and solving the problem of “long distance reordering” [29].
>
> [29] Sudoh, K., Duh, K., Tsukada, H., Hirao, T., & Nagata, M. (2010, July). Divide and translate: improving long distance reordering in statistical machine translation. In Proceedings of the Joint Fifth Workshop on Statistical Machine Translation and MetricsMATR (pp. 418-427). Association for Computational Linguistics.

[Zhang (2017)](https://syncedreview.com/2017/08/17/history-and-frontier-of-the-neural-machine-translation) cautions however that this model has some performance issues:

> However, the problem of “exploding/vanishing gradient” [28] makes RNN hard to actually handle the long distance dependencies; accordingly, the NMT model did not achieve a good performance at the beginning.
> 
>  [28] Pascanu, R., Mikolov, T., & Bengio, Y. (2013, February). On the difficulty of training recurrent neural networks. In International Conference on Machine Learning (pp. 1310-1318).

#### Bengio et. al. (2013) - Analysis of RNN Optimizations

In December 2012 (cited as Bengio 2013), *Advances in Optimizing Recurrent Networks* was [published](https://arxiv.org/abs/1212.0901). This survey paper documents the pain points and advancements of training RNNs.

The abstract goes on to note:

>  Although recurrent networks are extremely powerful in what they can in principle represent in terms of modelling sequences,their training is plagued by two aspects of the same issue regarding the learning of long-term dependencies. Experiments reported here evaluate the use of clipping gradients, spanning longer time ranges with leaky integration, advanced momentum techniques, using more powerful output probability models, and encouraging sparser gradients to help symmetry breaking and credit assignment. The experiments are performed on text and music data and show off the combined effects of these techniques in generally improving both training and test error.


#### Mnih et. al. (2014) - RNNs outperform CNNs

June 2014, Mnih et, al. [publish](https://arxiv.org/abs/1406.6247) *Recurrent Models of Visual Attention*. In the paper, Mnih et. al. make the case for RNNs over CNNs in the context of image processing. The abstract reads:

> Applying convolutional neural networks to large images is computationally expensive because the amount of computation scales linearly with the number of
image pixels. We present a novel recurrent neural network model that is capable of extracting information from an image or video by adaptively selecting
a sequence of regions or locations and only processing the selected regions at
high resolution. Like convolutional neural networks, the proposed model has a
degree of translation invariance built-in, but the amount of computation it performs can be controlled independently of the input image size. While the model
is non-differentiable, it can be trained using reinforcement learning methods to
learn task-specific policies. We evaluate our model on several image classification
tasks, where it significantly outperforms a convolutional neural network baseline
on cluttered images, and on a dynamic visual control problem, where it learns to
track a simple object without an explicit training signal for doing so.

The author then goes on to highlight that they are not the first to apply attention to deep learning, but I believe they are the first to apply it to an RNN.

> Our work is perhaps the most similar to the other attempts to implement attentional processing in a
deep learning framework [6, 14, 17]. Our formulation which employs an RNN to integrate visual
information over time and to decide how to act is, however, more general, and our learning procedure
allows for end-to-end optimization of the sequential decision process instead of relying on greedy
action selection. We further demonstrate how the same general architecture can be used for efficient
object recognition in still images as well as to interact with a dynamic visual environment in a
task-driven way.


Additionally they propose The Recurrent Attention Model (RAM). In this architecture, the model automatically adjusts the "bandwidth" or the "vector length" being observed durign the training process.

#### Cho et. al. (2014) - Joint Training Encoder-Decoder RNNs
In June 2014, Cho et. al. [published](https://arxiv.org/abs/1406.1078) *Learning Phrase Representations using RNN Encoder-Decoder for Statistical Machine Translation*. The abstract reads:

> In this paper, we propose a novel neural network model called RNN Encoder-Decoder that consists of two recurrent neural networks (RNN). One RNN encodes a sequence of symbols into a fixed-length vector representation, and the other decodes the representation into another (variable length) sequence of symbols. The encoder and decoder of the proposed model are jointly trained to maximize the conditional probability of a target sequence given a source sequence. The performance of a statistical machine translation system is empirically found to improve by using the conditional probabilities of phrase pairs computed by the RNN Encoder-Decoder as an additional feature in the existing log-linear model. Qualitatively, we show that the proposed model learns a semantically and syntactically meaningful representation of linguistic phrases.

An important feature of this model is it's attention mechanism which it claims functions similar to the mechanism provided by LSTM but is simpler to compute and impliment:

> the update gate controls how
much information from the previous hidden state
will carry over to the current hidden state. This
acts similarly to the memory cell in the LSTM
network and helps the RNN to remember long-
term information. Furthermore, this may be con-
sidered an adaptive variant of a leaky-integration
unit (Bengio et al., 2013).

**Note**: By unit, Cho is referring to "Hidden Unit" or to a node in the hidden layer.


#### Sutskever et. al. (2014) - Joint LSTM Encoder-Decoder

In september 2014, Sutskever et. al. [published](https://arxiv.org/abs/1409.3215) *Sequence to Sequence Learning with Neural Networks*

Their paper presents a novel unsupervised approach to sequence learning (I.e. learning how to map sequences). Being unsupervised, the approach made minimal structural assumptions about the sequences. 

**Note**: I use the term method or approach, rather than model, because the methodology described in the paper uses the encoder-decoder framework and thus multiple models. For the encoder, the authors use a multilayered Long Short-Term Memory (LSTM) to map the input sequence to a vector of a fixed dimensionality, and then another deep LSTM to decode the target sequence from the vector.

**Note**: This approach this still hindered by the fixed vector problem.

Bahdanau et. al (2014) - Variable Length Embedding

In Sept 2014, Bahdanau, Cho, Benjio. [published](https://arxiv.org/abs/1409.0473) *Neural Machine Translation by Jointly Learning to Align and Translate* and propose an encoder-decoder which is not using a fixed length intermediary but a:

> an extension to the encoder–decoder model which learns to align and translate jointly. Each time the proposed model generates a word in a translation, it (soft-)searches for a set of positions in a source sentence where the most relevant information is concentrated. The model then predicts a target word based on the context vectors associated with
these source positions and all the previous generated target words.
>
> The most important distinguishing feature of this approach from the basic encoder–decoder is that
it does not attempt to encode a whole input sentence into a single fixed-length vector. Instead, it encodes the input sentence into a sequence of vectors and chooses a subset of these vectors adaptively while decoding the translation. This frees a neural translation model from having to squash all the information of a source sentence, regardless of its length, into a fixed-length vector. We show this allows a model to cope better with long sentences.

This paper introduces the concept of a Context Vector which replaces the prior fixed length intermediary representations.

#### (2015) - OpenAI Founded

> OpenAI was founded in 2015 as a nonprofit research organization by Altman, Elon Musk, Peter Thiel, and LinkedIn cofounder Reid Hoffman, among other tech leaders.
>
> [vice](https://www.vice.com/en/article/5d3naz/openai-is-now-everything-it-promised-not-to-be-corporate-closed-source-and-for-profit)

#### Ramachandran et al. (2016) - Initializing With Pretrained Weights
> Ramachandran et al. (2016) extends Dai and Le (2015) by proposing a pre-training method to improve the accuracy of sequence to sequence (seq2seq) models. The encoder and decoder of the seq2seq model is initialized with the pre-trained weights (as opposed to random weights, which are resolved based on the input and output sequence corpus).
>
> [ Liu et. al. (2020) - A Survey on Contextual Embeddings](https://arxiv.org/abs/2003.07278)

#### Vaswani et al. (2017) - Transformer Architecture
In June 2017, the transformer architecture was [published](https://arxiv.org/abs/1706.03762) by Vaswani et al. This new architecture incorporated and expanded upon the prior advancements with respect to attention. This model based solely on attention mechanisms, dispenses with recurrence and convolutions entirely.

>  has been shown to better capture global dependencies from the inputscompared to its alternatives, e.g. recurrent networks, and perform strongly on a range of sequence learning tasks, such as machine translation (Vaswani et al., 2017) and document generation (Liu et al., 2018).
>
> [ Liu et. al. (2020) - A Survey on Contextual Embeddings](https://arxiv.org/abs/2003.07278)

This model is discussed in further detail in the [Transformer notebook](Transformers.ipynb).

#### Peters et al. (2018) - Bidirectional Training

> The ELMo model (Peters et al., 2018) generalizes traditional word embeddings by extracting context-dependent representations from a bidirectional language model.
>
> [ Liu et. al. (2020) - A Survey on Contextual Embeddings](https://arxiv.org/abs/2003.07278)

The paper is titled *Deep contextualized word representations* and can be found [here](https://arxiv.org/abs/1802.05365).

#### Radford et al. (2018) - GPT

> GPT adopts a two-stage learning paradigm: (a) nsupervised pre-training using a language modelling objective and (b) supervised fine-tuning.
>
> The goal is to learn universal representations transferable to a wide range of downstream tasks.
>
> [ Liu et. al. (2020) - A Survey on Contextual Embeddings](https://arxiv.org/abs/2003.07278)

The Radford and the rest of the OpenAI team published *Improving Language Understanding
by Generative Pre-Training* on June 11, 2018 which can be found [here](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf)

GPT is a proprietary model owned by OpenAi.

#### Devlin et al. (2018) - BERT - Masked Language Modeling (MLM)

ELMo concatenates representations from the forward and backward LSTMs without considering the interactions between the left and right contexts. GPT and GPT-2 use a left-to-right decoder, where every token can only attend to its left context. These architectures are sub-optimal for sentence-level tasks,
e.g. named entity recognition and sentiment analysis, as it is crucial to incorporate contexts from
both directions. 

BERT proposes a masked language modelling (MLM) objective, where some of the tokens of a input sequence are randomly masked, and the objective is to predict these masked positions taking the corrupted sequence as input. BERT applies a Transformer encoder to attend to bi-directional contexts during pre-training.

[ Liu et. al. (2020) - A Survey on Contextual Embeddings](https://arxiv.org/abs/2003.07278)

#### Radford et al. (2019) GPT-2

GPT2 follows a similar architecture to the original GPT.

It trains on a significantly larger corpus named WebText which (scraped from reddit posts and coresponding outbound links) which inherantly exhibits instances of text formatted in a question and answer style structure as well as summarization information. This results in the model being ablt to solve a wide variety of NLP tasks without explicit supervision.

Like GPT, GPT-2 uses a left-to-right decoder rather than a bi-directional encoder like ELmo.

[ Liu et. al. (2020) - A Survey on Contextual Embeddings](https://arxiv.org/abs/2003.07278)

In the original paper the authors compare GPT-2 to GPT and BERT highlighting that GPT-2 is a "larger" model meaning that the neural network is larger and thus more weights need to be trained. The GPT-2 model is said to have ~1.5 Billion parameters.

GPt-2 was published February 2019 in a paper titled *Language Models are Unsupervised Multitask Learners* and can be found [here](https://d4mucfpksywv.cloudfront.net/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)

#### (2018 - 2019) - BERT Variants 
Variants of BERT including EARNIE, SpanBert, StructBert, RoBERTa, ALBERT further study and improve the objective and architecture of BERT

[ Liu et. al. (2020) - A Survey on Contextual Embeddings](https://arxiv.org/abs/2003.07278)


#### (Yang et al., 2019) - XLNet, Traversal Without Artificial Symbols

The XLNet model identifies two weaknesses of BERT:
1. BERT assumes conditional independence of corrupted tokens.
2. The symbols such as [MASK] are introduced by BERT during pre-training, yet they never occur in real data, resulting in a discrepancy between pre-training and fine-tuning.

XLNet proposes a new auto-regressive method based on permutation language modelling (PLM) (Uria et al., 2016) without introducing any new symbols. 

XLNet further adopts two-stream self-attention and Transformer-XL (Dai et al., 2019) to take into account the target positions and learn longrange dependencies, respectively

[ Liu et. al. (2020) - A Survey on Contextual Embeddings](https://arxiv.org/abs/2003.07278)


#### Clark et al. (2019) - ELECTRA - Traversing Via Imputation

> Compared to BERT, ELECTRA (Clark et al., 2019) proposes a more effective pretraining method. Instead of corrupting some positions of inputs with [MASK], ELECTRA replaces some tokens of the inputs with their plausible alternatives sampled from a small generator network. ELECTRA trains a discriminator to predict whether each token in the corrupted input was replaced by the generator or not. The pre-trained discriminator can then be used in downstream tasks for fine-tuning, improving upon the pre-trained representation learned by the generator.

[ Liu et. al. (2020) - A Survey on Contextual Embeddings](https://arxiv.org/abs/2003.07278)


#### Lewis et al. (2019) - BART - Advanced MLM

For more information see the [BART notebook](BART.ipynb)

#### (March 2019) - OpenAI goes Closed-source

> The code behind both GPT-1 and GPT-2 has been officially released by OpenAI and is available on GitHub for any developer to utilize and make improvements on. The same cannot be said for GPT-3. Rather than deliver on their original promise listed in their mission statement claiming that “[our code] will be shared with the world,” (OpenAI) OpenAI instead decided to not release the source code for GPT-3 and instead release the model in the form of service.
>
> [source](https://sites.imsa.edu/hadron/2021/02/03/openai-was-the-shift-to-closed-source-justified/)

> In (March) 2019, OpenAI became a for-profit company called OpenAI LP, controlled by a parent company called OpenAI Inc. The result was a “capped-profit” structure that would limit the return of investment at 100-fold the original sum. If you invested \\$10 million, at most you’d get \\$1 billion. Not exactly what I’d call capped.
>
> A few months after the change, Microsoft injected $1 billion. OpenAI’s partnership with Microsoft was sealed on the grounds of allowing the latter to commercialize part of the tech, as we’ve seen happening with GPT-3 and Codex.
>
> [source](https://onezero.medium.com/openai-sold-its-soul-for-1-billion-cf35ff9e8cd4)
>
> [source](https://www.technologyreview.com/2020/02/17/844721/ai-openai-moonshot-elon-musk-sam-altman-greg-brockman-messy-secretive-reality/)


#### Brown et al. (2020) - GPT-3 - 175 Billion Parameters

GPT-3 was announced in may and release in june 2020. 

In the [paper](https://arxiv.org/abs/2005.14165) the authors note that increasing the size of the language model improves is ability to perform downstream one-shot tasks:

> Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something which current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches. Specifically, we train GPT-3, an autoregressive language model with 175 billion parameters, 10x more than any previous non-sparse language model, and test its performance in the few-shot setting. For all tasks, GPT-3 is applied without any gradient updates or fine-tuning, with tasks and few-shot demonstrations specified purely via text interaction with the model. GPT-3 achieves strong performance on many NLP datasets, including translation, question-answering, and cloze tasks, as well as several tasks that require on-the-fly reasoning or domain adaptation, such as unscrambling words, using a novel word in a sentence, or performing 3-digit arithmetic. At the same time, we also identify some datasets where GPT-3's few-shot learning still struggles, as well as some datasets where GPT-3 faces methodological issues related to training on large web corpora. Finally, we find that GPT-3 can generate samples of news articles which human evaluators have difficulty distinguishing from articles written by humans. We discuss broader societal impacts of this finding and of GPT-3 in general.

#### (March 2022) - GPT-3.5 - Enhancements And Newer Data

In March 2022, OpenAI made available new versions of GPT-3 which were trained on newer data sets (including text as well as code) spannign up to June 2021. Additionally, the public api added new features like edit and insert.

There were several models/releases included in the 3.5 family:
- gpt-3.5-turbo (chat)
- text-davinci-002 (text completion)
- text-davinci-003 (text completion)

https://en.wikipedia.org/wiki/GPT-3

> GPT-3.5 is an upgraded version of GPT-3 with fewer parameters that includes a fine-tuning process for machine learning algorithms. The fine-tuning process involves reinforcement learning with human feedback, which helps to improve the accuracy and effectiveness of the algorithms. Additionally, GPT-3.5 is designed to work within policies based on ethical human values, ensuring that the AI systems it powers are safe and reliable for human use.
>
> [source](https://www.iffort.com/blog/2023/03/31/gpt-3-vs-gpt-3-5)

> Instead of releasing GPT-3.5 in its fully trained form, OpenAI utilized it to develop several systems specifically optimized for various tasks, all accessible via the OpenAI API. One of these, text-davinci-003, is said to handle more intricate commands than models constructed on GPT-3 and produce higher quality, longer-form writing.
>
> OpenAI data scientist Jan Leike stated that text-davinci-003 is comparable to InstructGPT, a series of GPT-3-based models that OpenAI introduced earlier this year. These models are designed to minimize the generation of problematic text, like toxic or highly biased content, while better adhering to a user’s intentions.
>
> https://blog.accubits.com/gpt-3-vs-gpt-3-5-whats-new-in-openais-latest-update/#What%E2%80%99s-different-in-GPT-3.5

#### (March 2023) - GPT 3.5 Turbo

According to this [article](https://www.ankursnewsletter.com/p/gpt-4-gpt-3-and-gpt-35-turbo-a-review):

> GPT-3.5 Turbo, released on March 1st, 2023, is an improved version of GPT 3.5 and GPT-3 Davinci.
>
> ...
>  
> OpenAI utilized a development technique known as Reinforcement Learning from Human Feedback (RLHF) when developing GPT-3.5 Turbo.
>
> ...
>
> GPT-3.5 Turbo provides many of the same capabilities as GPT-3. However, GPT-3.5 Turbo proved to be capable of answering much more versatile questions and acting on a wider range of commands. GPT-3.5 Turbo is also less likely to have hallucinatory responses.
>
> ...

#### (November 2022) - ChatGPT - User Facing Chat Bot

According to this [article](https://www.ankursnewsletter.com/p/gpt-4-gpt-3-and-gpt-35-turbo-a-review):
> ChatGPT ... was released on November 30th, 2022.
>
> ChatGPT, a web-browser application based off of a model from the GPT-3.5 series

According to this [article's](https://www.technologyreview.com/2023/03/03/1069311/inside-story-oral-history-how-chatgpt-built-openai) summary of an interview with the model's creators:

> The basic idea is to take a large language model with a tendency to spit out anything it wants—in this case, GPT-3.5—and tune it by teaching it what kinds of responses human users actually prefer.
>
> ...
> 
> Since November, OpenAI has already updated ChatGPT several times. The researchers are using a technique called adversarial training to stop ChatGPT from letting users trick it into behaving badly (known as jailbreaking). This work pits multiple chatbots against each other: one chatbot plays the adversary and attacks another chatbot by generating text to force it to buck its usual constraints and produce unwanted responses. Successful attacks are added to ChatGPT’s training data in the hope that it learns to ignore them.    


The original release of ChatGPT was based on the GPT-3 family models but has since been updated to support newer version (GPT-4) as well.

While the GPT family of models are geared towards researchers and developers, ChatGPT is geared towared uses. Through the Web UI or API, users can now interract with the GPT models in a much friendlier way.

This marked the beginning of a monumental shift in the worlds perception of this technology.

#### (March 2023) - GPT-4 - 100 trillion parameters

In addition to it's large size, GPT-4 bosts the following enhancements:

- Improved model alignment — the ability to follow user intention
- Lower likelihood of generating offensive or dangerous output
- Increased factual accuracy
- Better steerability — the ability to change behavior according to user requests
- Internet connectivity – the latest feature includes the ability to search the Internet in real-time

[source](https://www.forbes.com/sites/bernardmarr/2023/05/19/a-short-history-of-chatgpt-how-we-got-to-where-we-are-today/?sh=515ef82f674f)

According to this [article](https://www.ankursnewsletter.com/p/gpt-4-gpt-3-and-gpt-35-turbo-a-review) we see that GPT-4 vastly outperforms ChatGPT and the GPT-3.5 series:

><center><img src='./images/chatgpt_performance_on_bar_exam.png' style="width:75%"></center>


